# Homework 3 — Classification

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
# !wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
# cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = pd.read_csv(url) #[cols]

cat_features = ['lead_source', 'industry', 'employment_status', 'location']
num_features = ['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score']



In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
df.converted.value_counts(normalize=True)

converted
1    0.5766
0    0.4234
Name: proportion, dtype: float64

In [6]:
df[cat_features].nunique()

lead_source          5
industry             6
employment_status    4
location             5
dtype: int64

In [7]:
df[cat_features] = df[cat_features].fillna('NA')
df[num_features] = df[num_features].fillna(0)

## Q1. What is the most frequent observation (mode) for the column industry?

In [8]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [9]:
df['industry'].mode()

0    technology
Name: industry, dtype: object

## Q2. What are the two features that have the biggest correlation?

In [10]:
df.corr(numeric_only=True)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


answer: interaction_count vs lead_score

## Split

In [11]:
from sklearn.model_selection import train_test_split

df_full = df
# df = df[cat_features + num_features]
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


## Q3.  Which of these variables has the biggest mutual information score?

In [12]:
from sklearn.metrics import mutual_info_score

target = 'converted'

scores = {}
for col in cat_features:
    score = mutual_info_score(df_train[col].astype(str), df_train[target].astype(str))
    scores[col] = round(score, 2)

# Convertir a una serie de pandas ordenada de mayor a menor
results = pd.Series(scores).sort_values(ascending=False)

results

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.

    To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

## Q4. What accuracy did you get?

In [13]:
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train[num_features + cat_features].to_dict(orient='records')
dv = DictVectorizer(sparse=False)


In [14]:
X_train = dv.fit_transform(train_dicts)

dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [15]:
val_dicts = df_val[num_features + cat_features].to_dict(orient='records')
X_val = dv.transform(val_dicts)


In [16]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [17]:
y_train = df_train['converted']

In [18]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [19]:
from sklearn.metrics import accuracy_score

y_val_predicted = model.predict(X_val)
y_val = df_val['converted']
accuracy = accuracy_score(y_val, y_val_predicted)
print(round(accuracy,2))

0.64


In [20]:

y_train_predicted = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_predicted)
print(train_accuracy)

0.625


## Q5. Feature elimination

In [21]:
def val_accuracy(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

all_features = num_features + cat_features
original_accuracy = val_accuracy(all_features)

diffs = {f: original_accuracy - val_accuracy([x for x in all_features if x != f]) for f in all_features}
pd.Series(diffs).round(4).sort_values()

annual_income              -0.079
industry                    0.000
employment_status           0.000
location                    0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

## Q6. Regularization

In [22]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(val_accuracy(all_features, C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
